In [1]:
import os
import torch
import torch.nn as nn
import json

from config import get_config
from train_utils.gpu_utils import get_device_summary
from data.loader import get_dataloaders
from models.model import create_model
from train_utils.resume import init_resume_state
from train_utils.resume import fill_trackers_from_history
from train_utils.resume import load_pretrained_model
from train_utils.training_loop import run_training_loop
from train_utils.scheduler_utils import create_scheduler
from train_utils.training_summary import finalize_training_summary
from train_utils.training_summary import print_best_model_summary
from train_utils.plot_metrics import plot_train_val_metrics
from train_utils.plot_metrics import plot_loss_accuracy
from train_utils.plot_metrics import plot_confusion_matrices


In [2]:
cfg=get_config(config_path="experiments/exp_01_agg500_testing_vit/config/" \
"vit_tiny_patch16_224_gaussian_bs32_ep50_lr1e-04_p12_ds7200000_g500_sched-RLRP_preload_p4.yml")

# cfg=get_config()
print(json.dumps(vars(cfg), indent=2))

[INFO] Config Path: experiments/exp_01_agg500_testing_vit/config/vit_tiny_patch16_224_gaussian_bs32_ep50_lr1e-04_p12_ds7200000_g500_sched-RLRP_preload_p4.yml
[INFO] Detected native Ubuntu host: DS044955
[INFO] Using dataset root: /home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/
[INFO] Using dataset_size from config: 14000
{
  "model_tag": "ViT_tiny_patch16_224_gaussian_lrp_12_rlrp_4",
  "backbone": "vit_gaussian",
  "batch_size": 32,
  "epochs": 50,
  "learning_rate": 0.0001,
  "patience": 12,
  "input_shape": [
    1,
    32,
    32
  ],
  "global_max": 121.79151153564453,
  "dataset_root_dir": "/home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/",
  "train_csv": "/home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/metadata/c0005_alpha_q0_agg500_14000jets_5folds/c0005_alpha_q0_agg500_14000jets_fold0_train.csv",
  "val_csv": "/home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT

In [3]:
os.makedirs(cfg.output_dir, exist_ok=True)
print(f"[INFO] Saving all outputs to: {cfg.output_dir}")

[INFO] Saving all outputs to: experiments/exp_01_agg500_testing_vit/training_output/ViT_tiny_patch16_224_gaussian_lrp_12_rlrp_4_bs32_ep50_lr1e-04_ds14000_g1_sched_ReduceLROnPlateau_weighted_lossenergy_loss_output_1.0_alpha_output_1.0_q0_output_1.0/fold_0


In [4]:
device= get_device_summary()

=== NVIDIA-SMI ===
Failed to initialize NVML: Driver/library version mismatch
NVML library version: 535.288

[INFO] PyTorch version:       2.5.1
[INFO] Using device: cpu
[INFO] CUDA available:        False
[INFO] CUDA toolkit version:  12.1
[INFO] Device count:          0
[INFO] Only CPU is available.


/home/arsalan/miniconda3/envs/pytorch/lib/python3.9/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /opt/conda/conda-bld/pytorch_1729647350383/work/c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/home/arsalan/miniconda3/envs/pytorch/lib/python3.9/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [5]:
print(cfg.train_csv)

/home/arsalan/wsu_grid_rs_grp_mljet/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500/metadata/c0005_alpha_q0_agg500_14000jets_5folds/c0005_alpha_q0_agg500_14000jets_fold0_train.csv


In [6]:
# Data
train_loader, val_loader, test_loader = get_dataloaders(cfg, device=device)

[INFO] Training samples: 11200
[INFO] Validation samples: 2800
[INFO] Test samples: 2800
[INFO] Length of training dataloader: 350
[INFO] Length of validation dataloader: 88
[INFO] Length of test dataloader: 88


In [7]:
# Model and optimizer
model, optimizer = create_model(cfg.backbone, cfg.input_shape, cfg.learning_rate)
model.to(device)

Using ViT model: vit_tiny_patch16_224, pretrained: False


ViTClassifier(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=192, out_features=768, bias=True)
          (act): GELU(approximate='none

In [8]:
if torch.cuda.device_count() > 1:
    print(f"Parallelizing model across {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)
elif torch.cuda.device_count() == 1:
    print("No parallelization, using single GPU")
elif torch.cuda.device_count() == 0:
    print("No GPU available, using CPU")

No GPU available, using CPU


In [9]:
scheduler = create_scheduler(optimizer, cfg, train_loader=train_loader)

In [10]:
criterion = {
    # 'energy_loss_output': nn.BCELoss(),
    'energy_loss_output': nn.BCEWithLogitsLoss(),
    'alpha_output': nn.CrossEntropyLoss(),
    'q0_output': nn.CrossEntropyLoss()
}
print(f"[INFO] Loss functions:{criterion}")

[INFO] Loss functions:{'energy_loss_output': BCEWithLogitsLoss(), 'alpha_output': CrossEntropyLoss(), 'q0_output': CrossEntropyLoss()}


In [11]:
print(f"[INFO] Init Training Trackers")
train_loss_energy_list, train_loss_alpha_list, train_loss_q0_list, train_loss_list = [], [], [],[]
train_acc_energy_list, train_acc_alpha_list, train_acc_q0_list, train_acc_list = [], [], [], []

print(f"[INFO] Init Validation Trackers")
val_loss_energy_list, val_loss_alpha_list,val_loss_q0_list,val_loss_list = [], [], [], []
val_acc_energy_list, val_acc_alpha_list,val_acc_q0_list ,val_acc_list = [],[],[],[]

[INFO] Init Training Trackers
[INFO] Init Validation Trackers


In [12]:
model, optimizer, start_epoch, best_acc, early_stop_counter, best_epoch, best_metrics, training_summary, all_epoch_metrics,summary_status = init_resume_state( model, optimizer, device,cfg)

[INFO] Init Resume/Training Parameters
[INFO] Starting fresh training run by initializing training summary
[INFO] 📄 Training summary saved to: experiments/exp_01_agg500_testing_vit/training_output/ViT_tiny_patch16_224_gaussian_lrp_12_rlrp_4_bs32_ep50_lr1e-04_ds14000_g1_sched_ReduceLROnPlateau_weighted_lossenergy_loss_output_1.0_alpha_output_1.0_q0_output_1.0/fold_0/training_summary.json


In [13]:
fill_trackers_from_history(
    all_epoch_metrics,
    train_loss_energy_list, train_loss_alpha_list, train_loss_q0_list, train_loss_list,
    train_acc_energy_list, train_acc_alpha_list, train_acc_q0_list, train_acc_list,
    val_loss_energy_list, val_loss_alpha_list, val_loss_q0_list, val_loss_list,
    val_acc_energy_list, val_acc_alpha_list, val_acc_q0_list, val_acc_list,
    summary_status, best_epoch
)

In [14]:
preload_path = getattr(cfg, "preload_model_path", None)
print(f"[INFO] Preloading model from: {preload_path}")


[INFO] Preloading model from: None


In [15]:
model, preloaded = load_pretrained_model(model, device, cfg)


[INFO] No preload_model_path specified — skipping preload.


In [16]:
# for testing
# train_metrics = train_one_epoch(train_loader, model, criterion, optimizer, device)
# print(f"[INFO] Training metrics: {train_metrics}")

In [ ]:
best_epoch,best_acc,best_metrics=run_training_loop(
                      cfg,train_loader,val_loader,
                      device, model,criterion,
                      optimizer,scheduler,
                      start_epoch,early_stop_counter,
                      best_acc,best_metrics,best_epoch,
                      train_loss_list,
                        train_loss_energy_list,
                        train_loss_alpha_list,
                        train_loss_q0_list,
                        train_acc_list,
                        train_acc_energy_list,
                        train_acc_alpha_list,
                        train_acc_q0_list,
                        val_loss_list,
                        val_loss_energy_list,
                        val_loss_alpha_list,
                        val_loss_q0_list,
                        val_acc_list,
                        val_acc_energy_list,
                        val_acc_alpha_list,
                        val_acc_q0_list,
                        all_epoch_metrics)

[INFO] Epoch 1/50


Training:  17%|█▋        | 59/350 [04:31<22:09,  4.57s/it]

In [ ]:
finalize_training_summary(
    summary=training_summary,
    best_epoch=best_epoch,
    best_acc=best_acc,
    best_metrics=best_metrics,
    output_dir=cfg.output_dir
)
print_best_model_summary(
    best_epoch=best_epoch,
    best_acc=best_acc,
    best_metrics=best_metrics
)

In [ ]:
plot_train_val_metrics(train_loss_list, val_loss_list, train_acc_list, val_acc_list, cfg.output_dir)
plot_loss_accuracy(train_loss_list,
                    train_loss_energy_list,
                    train_loss_alpha_list,
                    train_loss_q0_list,
                    train_acc_list,
                    train_acc_energy_list,
                    train_acc_alpha_list,
                    train_acc_q0_list,
                    cfg.output_dir,
                    title="Train Loss and Accuracy per Epoch")
plot_loss_accuracy(val_loss_list,
                    val_loss_energy_list,
                    val_loss_alpha_list,
                    val_loss_q0_list,
                    val_acc_list,
                    val_acc_energy_list,
                    val_acc_alpha_list,
                    val_acc_q0_list,
                    cfg.output_dir,
                    title="Validation Loss and Accuracy per Epoch")

In [ ]:
plot_confusion_matrices(best_metrics, output_dir= f"{cfg.output_dir}/val", color_map="Oranges")

In [ ]:
alpha_names = ("0.2","0.3","0.4")
q0_names    = ("1.0","1.5","2.0","2.5")


In [ ]:
from train_utils.evaluate import evaluate
# experiment folder for artifacts
art_dir = os.path.join(cfg.output_dir,"val/prob_plots")
os.makedirs(art_dir, exist_ok=True)

alpha_hist_path      = os.path.join(art_dir, "alpha_pred_hist")      # .png/.pdf added by evaluate
alpha_avgprob_path   = os.path.join(art_dir, "alpha_avgprob")        # .png/.pdf added by evaluate
q0_avgprob_path      = os.path.join(art_dir, "q0_avgprob")           # .png/.pdf added by evaluate

metrics_val = evaluate(
    val_loader, model, criterion, device,
    loss_weights=getattr(cfg, "loss_weights", None),
    # plots:
    make_alpha_fig=True,
    alpha_fig_path=str(alpha_hist_path),
    make_alpha_avgprob_fig=True,
    alpha_avgprob_fig_path=str(alpha_avgprob_path),
    make_q0_avgprob_fig=True,
    q0_avgprob_fig_path=str(q0_avgprob_path),
    alpha_class_names=alpha_names,
    q0_class_names=q0_names,
)

print("Saved images:")
print("  α_s histogram:        ", metrics_val.get("alpha_hist_path"))
print("  α_s avg-prob bars:    ", metrics_val.get("alpha_avgprob_hist_path"))
print("  α_s probabilities:     ", metrics_val.get("alpha_probs_csv"))
print("  α_s heatmap:          ", metrics_val.get("alpha_heatmap_path"))
print("  Q0  avg-prob bars:    ", metrics_val.get("q0_avgprob_hist_path"))
print("  Q0  probabilities:     ", metrics_val.get("q0_probs_csv"))
print("  Q0  heatmap:          ", metrics_val.get("q0_heatmap_path"))


In [ ]:
# (optional) test split as well
# try:
#     from train_utils.evaluate import evaluate
#     # experiment folder for artifacts
#     art_dir = os.path.join(cfg.output_dir,"test/prob_plots")
#     os.makedirs(art_dir, exist_ok=True)

#     alpha_hist_path      = os.path.join(art_dir, "alpha_pred_hist")      # .png/.pdf added by evaluate
#     alpha_avgprob_path   = os.path.join(art_dir, "alpha_avgprob")        # .png/.pdf added by evaluate
#     q0_avgprob_path      = os.path.join(art_dir, "q0_avgprob")           # .png/.pdf added by evaluate

#     metrics_test = evaluate(
#         test_loader, model, criterion, device,
#         loss_weights=getattr(cfg, "loss_weights", None),
#         # plots:
#         make_alpha_fig=True,
#         alpha_fig_path=str(alpha_hist_path),
#         make_alpha_avgprob_fig=True,
#         alpha_avgprob_fig_path=str(alpha_avgprob_path),
#         make_q0_avgprob_fig=True,
#         q0_avgprob_fig_path=str(q0_avgprob_path),
#         alpha_class_names=alpha_names,
#         q0_class_names=q0_names,
#     )

#     print("Saved images:")
#     print("  α_s histogram:        ", metrics_test.get("alpha_hist_path"))
#     print("  α_s avg-prob bars:    ", metrics_test.get("alpha_avgprob_hist_path"))
#     print("  α_s probabilities:     ", metrics_test.get("alpha_probs_csv"))
#     print("  α_s heatmap:          ", metrics_test.get("alpha_heatmap_path"))
#     print("  Q0  avg-prob bars:    ", metrics_test.get("q0_avgprob_hist_path"))
#     print("  Q0  probabilities:     ", metrics_test.get("q0_probs_csv"))
#     print("  Q0  heatmap:          ", metrics_test.get("q0_heatmap_path"))
#     have_test = True
# except NameError:
#     have_test = False